# Supplementary Table 18 — the effector gene list

The effector gene list itself, not the training loci it produces. `data/intermediate_files/EGL.parquet`
is `gs://genetics-portal-dev-analysis/yt4/2506_release/training_set/20250625_EGL_2506_0.95_otg_chembl.parquet`,
written by `01_EGL_preparation.ipynb` of `~/Projects/EGL_and_training_set/2506` and read by
`02_training_set.ipynb` as its `sgl` input. It is the union of three branches, de-duplicated on
`(targetId, diseaseId)`:

| branch                                | selection                                                                                                                                              | pairs  |
| ------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------ | ------ |
| ChEMBL Phase 3/4                      | release `evidence/sourceId=chembl`, `clinicalPhase in (3, 4)`                                                                                           | 25,685 |
| high-confidence platform associations | release `evidence`, `score >= 0.95`, `datasourceId in (eva, uniprot_variants, gene2phenotype, genomics_england, clingen, uniprot_literature, orphanet)` | 16,371 |
| previous Open Targets Genetics gold standards | `otg_gs_230511.json`, `highest_confidence in (High, Medium)`, ontology exploded                                                                 | 812    |

`europepmc` literature evidence is prepared in that notebook and deliberately not unioned in.

The union drops `datasourceId` and the High/Medium label, so the parquet is two columns wide —
`diseaseId`, `targetId`. The caption's contributing-source and confidence columns cannot be filled
from it; they would have to be reconstructed by re-running each branch with a source tag. See the
chapter README. What this notebook adds is the two label columns the release does carry: the gene
symbol from the target index and the disease name from the disease index.

This is the list as it goes into training. The labelled training set it produces — positives and
negatives, one row per credible-set/gene pair — is `ST19`, written by `06_l2g_training_set.ipynb`.
`03_l2g_tables.ipynb` used to write a sheet under the ST13 caption from the positive half of that
training set; it was the wrong artefact and was retired on 2026-09-02.

In [1]:
import pandas as pd
import pyarrow.dataset as pads

from manuscript_methods import paper

EGL = paper.baseline("EGL.parquet")
SHEET = paper.ROOT / "chapters/06-supplementary-tables/sheets/ST18_effector_gene_list.csv"

egl = pads.dataset(EGL, format="parquet").to_table(columns=["diseaseId", "targetId"]).to_pandas()
print(f"EGL pairs: {len(egl):,}")
print(f"distinct pairs: {egl.drop_duplicates().shape[0]:,}")
print(f"genes: {egl['targetId'].nunique():,} | diseases: {egl['diseaseId'].nunique():,}")

EGL pairs: 42,288
distinct pairs: 42,288
genes: 5,548 | diseases: 8,643


## Labels from the release

`approvedSymbol` from the target index and `name` from the disease index, both of the 25.06 release
the list was built against. A pair whose id is absent from either index keeps the id and gets an
empty label rather than being dropped — the sheet has to stay the effector gene list, not the part
of it the release still resolves.

In [2]:
genes = (
    pads.dataset(paper.release("target"), format="parquet")
    .to_table(columns=["id", "approvedSymbol"])
    .to_pandas()
    .rename(columns={"id": "targetId"})
    .drop_duplicates(subset="targetId")
)
diseases = (
    pads.dataset(paper.release("disease"), format="parquet")
    .to_table(columns=["id", "name"])
    .to_pandas()
    .rename(columns={"id": "diseaseId"})
    .drop_duplicates(subset="diseaseId")
)

labelled = egl.merge(genes, on="targetId", how="left").merge(diseases, on="diseaseId", how="left")
assert len(labelled) == len(egl), "the label joins changed the number of pairs"

unnamed = labelled.loc[labelled["name"].isna(), "diseaseId"]
print(f"genes with no symbol in the target index: {labelled['approvedSymbol'].isna().sum():,}")
print(f"rows with no disease name: {len(unnamed):,} over {unnamed.nunique():,} ids")
print(f"prefixes: {sorted(unnamed.str.split('_').str[0].unique())}")

genes with no symbol in the target index: 0
rows with no disease name: 81 over 18 ids
prefixes: ['CHEBI', 'EFO', 'GO', 'Orphanet']


In [3]:
egl_sheet = (
    labelled.rename(
        columns={
            "diseaseId": "EFO ID",
            "targetId": "Ensembl gene ID",
            "approvedSymbol": "Gene symbol",
            "name": "Disease name",
        }
    )[["EFO ID", "Ensembl gene ID", "Gene symbol", "Disease name"]]
    .drop_duplicates()
    .sort_values(["EFO ID", "Ensembl gene ID"])
    .reset_index(drop=True)
)
egl_sheet.to_csv(SHEET, index=False)

print(f"rows: {len(egl_sheet):,}")
print(f"genes: {egl_sheet['Ensembl gene ID'].nunique():,} | diseases: {egl_sheet['EFO ID'].nunique():,}")
egl_sheet.head()

rows: 42,288
genes: 5,548 | diseases: 8,643


,EFO ID,Ensembl gene ID,Gene symbol,Disease name
0,CHEBI_9150,ENSG00000130203,APOE,NaN
1,DOID_10113,ENSG00000100342,APOL1,trypanosomiasis
2,DOID_10113,ENSG00000113578,FGF1,trypanosomiasis
3,DOID_10113,ENSG00000115758,ODC1,trypanosomiasis
4,DOID_13406,ENSG00000106348,IMPDH1,pulmonary sarcoidosis


## Against the training set it produced

The 42,288 pairs are what goes into `02_training_set.ipynb`; the gold standard that comes out holds
only the pairs that matched a credible set of a qualified, replicated study, so the two sheets
cannot be the same artefact.

The comparison has to be made at the row level, not the exploded-pair level. A positive is flagged
by `array_contains(diseaseIds, sgl.diseaseId)`, so a study carrying several disease ids contributes
one matched pair and several co-occurring ones; exploding `diseaseIds` and intersecting with the
EGL therefore undercounts by construction. The assertion below is the one that means something:
**every** positive row has at least one of its disease ids in the EGL.

In [4]:
gold = (
    pads.dataset(str(paper.ROOT / "data/l2g_training_set/20250625_gentropy_paper_v1"), format="parquet")
    .to_table(columns=["geneId", "diseaseIds", "goldStandardSet"])
    .to_pandas()
)
rows = gold[gold["goldStandardSet"] == "positive"][["geneId", "diseaseIds"]].copy()
rows["key"] = list(zip(rows["geneId"], rows["diseaseIds"].map(tuple)))
rows = rows.drop_duplicates(subset="key")

pairs = set(map(tuple, egl[["targetId", "diseaseId"]].to_numpy()))
matched = [any((gene, disease) in pairs for disease in diseases) for gene, diseases in rows["key"]]
assert all(matched), "a positive gold-standard row has no disease id in the EGL"

exploded = (
    rows.explode("diseaseIds")
    .rename(columns={"diseaseIds": "diseaseId", "geneId": "targetId"})[["diseaseId", "targetId"]]
    .dropna()
    .drop_duplicates()
)
kept = exploded.merge(egl, on=["diseaseId", "targetId"], how="inner")

print(f"EGL pairs: {len(egl):,}")
print(f"positive gene-diseaseIds rows: {len(rows):,}, all matched to the EGL")
print(f"matched EGL pairs behind them: {len(kept):,} ({len(kept) / len(egl):.1%} of the EGL)")
print(f"co-occurring exploded pairs not in the EGL: {len(exploded) - len(kept):,}")

paper.save_results(
    "supplementary_table_effector_gene_list",
    {
        "egl_pairs": len(egl_sheet),
        "egl_genes": int(egl_sheet["Ensembl gene ID"].nunique()),
        "egl_diseases": int(egl_sheet["EFO ID"].nunique()),
        "egl_pairs_in_training_set": len(kept),
        "training_set_positive_rows": len(rows),
    },
)

EGL pairs: 42,288
positive gene-diseaseIds rows: 1,377, all matched to the EGL
matched EGL pairs behind them: 612 (1.4% of the EGL)
co-occurring exploded pairs not in the EGL: 1,092


'/Users/yt4/Projects/Gentropy-manuscript/results/supplementary_table_effector_gene_list.json'